In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import root
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 1. Динамика CR3BP
# ------------------------------------------------------------------
def cr3bp_equations(t, state, mu):
    x, y, vx, vy = state
    r1 = ((x + mu) ** 2 + y ** 2) ** 0.5
    r2 = ((x - 1 + mu) ** 2 + y ** 2) ** 0.5

    Ux = x - (1 - mu) * (x + mu) / r1 ** 3 - mu * (x - 1 + mu) / r2 ** 3
    Uy = y - (1 - mu) * y / r1 ** 3 - mu * y/ r2 ** 3

    return np.array([vx,
                     vy,
                     2 * vy + Ux,
                    -2 * vx + Uy])

# ------------------------------------------------------------------
# 2. Периодические условия (симметрия по Ox)
# ------------------------------------------------------------------
def F(z, x0, mu):
    vy0, T = z
    y_init = np.array([x0, 0.0,  0.0, vy0])

    sol = solve_ivp(lambda t, state: cr3bp_equations(t, state, mu), [0, T], y_init,  method="LSODA",
                   rtol=1e-12, atol=1e-12, t_eval=np.linspace(0, T, 500),)

    yT = sol.y[:, -1]          # состояние в t = T
    
    return np.hstack((np.sqrt(yT[1]**2 + (yT[0] - x0) ** 2), np.sqrt((yT[3] - vy0)**2 + yT[2]**2)))

# ------------------------------------------------------------------
# 3. Двухтельная (μ→0) начальная оценка
# ------------------------------------------------------------------
def twobody_guess(x0, mu):
    """
    Круговая орбита вокруг m1 в инерц. системе.
    В безразм. единицах GM1 ≈ 1 → v = sqrt(1/r)
    Переход в вращающуюся систему (ω=1): vy0 = v - x0
    """
    v_inert = np.sqrt(1.0 / x0)
    vy0 = v_inert - x0           # y-скорость в вращ. системе
    T   = 2*np.pi * x0/(np.sqrt((1- mu)/x0))           # первичная оценка периода
    return np.array([vy0, T])

# ------------------------------------------------------------------
# 4. Решение для одного (x0, μ)
# ------------------------------------------------------------------
def solve_orbit(x0, mu, guess=None):
    if guess is None:
        guess = twobody_guess(x0, mu)

    sol_orb = root(lambda z: F(z, x0, mu), guess, method='hybr',
               tol=1e-10)

    return sol_orb.x                  # [vy0, T]


In [ ]:
# ------------------------------------------------------------------
# 5. Построение всей семейства на сетке (μ, x0)
# ------------------------------------------------------------------
def build_family(mu_min=1e-7, mu_max=0.038,
                 N_mu=50, x0_min=0.05, x0_max=0.7, N_x=60):
    """
    Возвращает словарь:  data[(i_mu, i_x)] = (vy0, T)
    """
    mu_grid  = np.geomspace(mu_min, mu_max, N_mu)
    x0_grid  = np.linspace (x0_min, x0_max, N_x)

    data = {}
    # --- 1) сначала решаем для первого μ на всей линии x0
    mu0 = mu_grid[0]
    for i_x, x0 in enumerate(x0_grid):
        sol = solve_orbit(x0, mu0)              # двухтельная и корректор
        data[(0, i_x)] = sol                    # (vy0, T)
    
    # --- 2) продолжаем по μ, используя предыдущие решения как guess
    for i_mu in range(1, N_mu):
        mu_cur = mu_grid[i_mu]
        mu_prev = mu_grid[i_mu-1]

        for i_x, x0 in enumerate(x0_grid):
            guess = data[(i_mu-1, i_x)]         # (vy0, T) от предыдущего μ
            try:
                sol = solve_orbit(x0, mu_cur, guess)
            except RuntimeError:                # если сорвалось, пробуем «соседа» по x
                neighbour = data.get((i_mu, i_x-1), guess)
                sol = solve_orbit(x0, mu_cur, neighbour)

            data[(i_mu, i_x)] = sol

    return mu_grid, x0_grid, data

In [ ]:
mu_grid, x0_grid, family = build_family()

# Сохраним всё в numpy-файл
M = len(mu_grid); N = len(x0_grid)
vy0_mat = np.zeros((M, N))
T_mat   = np.zeros((M, N))
for i in range(M):
    for j in range(N):
        vy0_mat[i, j], T_mat[i, j] = family[(i, j)]

np.savez('quasi_circular_family.npz',
         mu=mu_grid, x0=x0_grid, vy0=vy0_mat, T=T_mat)

# Быстрый визуальный контроль – период против x0 для пары μ
for idx in (0, int(0.25*M), int(0.5*M), int(0.75*M), -1):
    plt.plot(x0_grid, T_mat[idx], label=f'μ={mu_grid[idx]:.2e}')
plt.xlabel('x0'); plt.ylabel('Period T'); plt.legend(); plt.show()

In [ ]:
x0 = 1.000e-07
mu0 = 0.05

In [ ]:
mu_min=1e-7
mu_max=0.038
N_mu=50
x0_min=0.05
x0_max=0.7
N_x=60

In [ ]:
"""
Возвращает словарь:  data[(i_mu, i_x)] = (vy0, T)
"""
mu_grid  = np.geomspace(mu_min, mu_max, N_mu)
x0_grid  = np.linspace (x0_min, x0_max, N_x)

data = {}
# --- 1) сначала решаем для первого μ на всей линии x0
mu0 = mu_grid[0]
sol_prev = None
for i_x, x0 in enumerate(x0_grid):
    sol = solve_orbit(x0, mu0)              # двухтельная и корректор
    data[(0, i_x)] = sol                    # (vy0, T)
    sol_prev = sol

In [ ]:
sol = solve_orbit(x0, mu0)   

In [ ]:
guess = twobody_guess(x0, mu0)

In [ ]:
sol_orb = root(lambda z: F(z, x0, mu0), guess, method='Krylov',
               tol=1e-10)

In [ ]:
sol_orb

In [ ]:
mu0 = 1.000e-07
x0 = 0.094
guess = twobody_guess(x0, mu0)


In [ ]:
mu0 = mu_grid[0]
for i_x, x0 in enumerate(x0_grid):
    sol_ivp = data[(0, i_x)]  
    y_init = np.array([x0, 0.0,  0.0, sol_ivp[0]])

    sol = solve_ivp(lambda t, state: cr3bp_equations(t, state, mu0), [0, sol_ivp[1]], y_init,  method="LSODA",
                   rtol=1e-12, atol=1e-12, t_eval=np.linspace(0, sol_ivp[1], 500),)
    
    plt.plot(sol.y[0,:], sol.y[1, :])
plt.axis('equal')
plt.grid()

In [ ]:
guess = twobody_guess(x0, mu0)
z = guess
vy0, T = z
y_init = np.array([x0, 0.0,  0.0, vy0])

sol = solve_ivp(lambda t, state: cr3bp_equations(t, state, mu), [0, T], y_init,  method="LSODA",
               rtol=1e-12, atol=1e-12, t_eval=np.linspace(0, T, 500),)


In [ ]:
T

In [ ]:

import matplotlib.pyplot as plt

In [ ]:
plt.plot(sol.y[0,:], sol.y[1, :])
plt.axis('equal')
plt.grid()

In [ ]:
sol = solve_orbit(x0, mu0)  

In [ ]:
F(z, x0, mu)

In [ ]:
sol = root(lambda z: F(z, x0, mu), guess, method='hybr',
               tol=1e-10)

In [ ]:
sol

In [ ]:
y_init = np.array([x0, 0.0,  0.0, sol.x[0]])

sol = solve_ivp(lambda t, state: cr3bp_equations(t, state, mu), [0, sol.x[1]], y_init,  method="LSODA",
               rtol=1e-12, atol=1e-12, t_eval=np.linspace(0, sol.x[1], 500),)

plt.plot(sol.y[0,:], sol.y[1, :])
plt.axis('equal')
plt.grid()

In [ ]:
import numpy as np

In [ ]:
4/1-1

In [ ]:
mu = 3.0393890e-6
for x0 in [0.63, 0.75, 0.82, 0.86, 0.88, 1.14, 1.17, 1.23, 1.32, 1.59]:
    T = (x0)/(np.sqrt((1-mu)/x0) - x0)
    R = 1+1/T
    print(T,R,Fraction(R).limit_denominator(6))

In [ ]:
mu = 3.0393890e-6
for x0 in [1.14, 1.17, 1.23, 1.32, 1.59]:
    T = (x0)/(np.sqrt((1-mu)/x0) - x0)
    R = 1+1/T
    print(T,R,Fraction(R).limit_denominator(6))

In [ ]:
from fractions import Fraction

In [ ]:
for k in [1.000191049, 1.853230181, 2.884139381, 3.939061919, 4.731132896]:
    print(Fraction(k+1).limit_denominator(100))

In [ ]:
mu = 3.0393890e-6
T = 2.pi*(x0)/(np.sqrt((1-mu)/x0) - x0)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(np.sqrt((1-mu)/(x + mu)) - (x + mu)) + (0.47084948*x + (x - 0.21319899)*(1.1044086*x/np.cos(0.24954179 - 2.0628507/np.log(x)) - 0.0420995)*(np.sin(2.117359/np.log(x)) + 0.014023513)/(x - 0.92994696))*1e-5"

litera = "C"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_y", j=18, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "2 * np.pi * (x0 + mu)/(np.sqrt((1-mu)/(x0 + mu)) - (x0 + mu))+ (x0**14.47737*(x0 - 0.7659898)*(654.6419*np.sin(289.54312/x0) + 246.73122)*(np.sin(6.09837867894604*x0/(x0 - 1.0300114)) - 0.31069863)*np.cos(718.70135*x0)/np.sin(289.54312/x0)) * 1e-2"

litera = "D"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="T", j=10, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(np.sqrt((1-mu)/x) - x) +(-0.015745996*x/((x - 0.04343827/(x - 1.5630211))*np.log(x - 0.3106394)*np.log(x - 0.1418749)) + 0.26363623 - 0.075398974/(x - 1.165357) - 0.05681477/(x - 1.2188393)) * 1e-5"

litera = "C"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_y", j=19, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "2 * np.pi * (x)/(np.sqrt((1-mu)/x) - x) + (-0.013422184538546/np.log(1.03323023417647*x - 0.35217838513347) + 0.452854*np.exp(-x*(10.027951*x + 14.331656758621*np.sin(1.3243008/(x - x**x) - 0.74299186/x))) - 0.08211122462468/(np.log(x) - 0.07767904) + 0.3439773016164/x)*1e-2"

litera = "D"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="T", j=11, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
for i, v in consts:
    print(v)